# E1 · Limpieza y hallazgos del tablero
Pedidos de Salomé del 5 de febrero de 2020 al 24 de octubre de 2026.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent
sys.path += [str(RAIZ / "services/tablero"), str(RAIZ / "services/worker")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import analitica as an, modelos_e2 as m2
from limpieza import limpiar, validar_limpio
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Datos crudos
Así llegan a `raw/`: una fila por producto dentro de cada pedido, con los errores de copiar los chats a mano.

In [2]:
crudo = pd.read_csv(RAIZ / "data/historial/historial_hasta_2026-10-24.csv", dtype=str)
print(crudo.shape)
crudo.sample(8, random_state=3)

(2619, 13)


,pedido_id,cliente_id,fecha_pedido,hora_pedido,fecha_entrega,producto,categoria,unidades,precio_unitario,total,canal,entrega,metodo_pago
301,1219,C0083,2021-09-04,21:49,2021-09-05,Caja de brownies (6 pz),Galletas y brownies,1,130,130,WhatsApp,Recoge,Transferencia
1111,1827,C0119,2023-08-03,12:42,2023-08-05,Rebanada tres leches,Rebanadas,1,55,65,WhatsApp,Domicilio,Efectivo
1243,1921,C0394,2023-11-02,20:34,2023-11-03,Cheesecake frutos rojos entero,Pasteles enteros,NaN,490,490,WhatsApp,Recoge,Efectivo
1722,2286,C0348,2024-11-02,21:16,2024-11-03,Rebanada cheesecake frutos rojos,Rebanadas,1,62,62,Instagram,Recoge,Efectivo
318,1232,C0091,2021-09-24,23:12,2021-09-25,Rebanada cheesecake frutos rojos,Rebanadas,2,48,96,WhatsApp,Domicilio,Transferencia
586,1444,C0221,2022-05-28,19:42,2022-05-28,Pastel tres leches entero,Pasteles enteros,1,450,450,WhatsApp,Recoge,Transferencia
1234,1916,C0391,2023-10-31,20:38,2023-10-31,Pan de muerto (4 pz),Temporada,1,150,150,Instagram,Domicilio,Transferencia
355,1261,C0145,2021-10-31,20:19,2021-11-03,Cheesecake frutos rojos entero,Pasteles enteros,1,400,400,Instagram,Recoge,Transferencia


## 2. Limpieza
`limpiar()` es el mismo módulo que corre el worker. El reporte dice cuántos renglones se tocaron por tipo de error.

In [3]:
limpio, rep = limpiar(crudo)
r = rep.como_dict()
print(r["crudos"], "->", r["limpios"], "renglones; irrecuperables:", r["irrecuperables"])
pd.Series(r["conteos"]).loc[lambda s: s > 0].sort_values(ascending=False).to_frame("renglones")

2619 -> 2606 renglones; irrecuperables: 0


,renglones
formato_producto,38
formato_canal,26
fecha_formato,26
hora_vacia_sin_imputar,13
duplicados,13
total_inconsistente,10
unidades_texto,10
unidades_vacias,7
precio_vacio,7
unidades_atipicas,5


In [4]:
print("validación del archivo limpio:", validar_limpio(limpio) or "sin problemas")

validación del archivo limpio: sin problemas


## 3. Indicadores

In [5]:
df = an.preparar(limpio)
k = an.kpis(df)
{c: k[c] for c in ["ventas", "pedidos", "ticket", "ticket_mediana", "estrella", "estrella_pct", "ult7", "ant7", "var7"]}

{'ventas': np.int64(786227),
 'pedidos': 1958,
 'ticket': np.float64(401.54596527068435),
 'ticket_mediana': np.float64(360.0),
 'estrella': 'Cheesecake frutos rojos entero',
 'estrella_pct': np.float64(0.3511517666017575),
 'ult7': np.int64(4172),
 'ant7': np.int64(5115),
 'var7': np.float64(-0.18435972629521014)}

## 4. Títulos de las seis gráficas (se calculan de los datos)

In [6]:
t = an.titulos(df)
for c in ["lineas", "histograma", "calor", "dispersion", "espectro", "barras", "dia_semana"]:
    print(f"{c:11s}", t[c])

lineas      Septiembre 2026 vendió 31% más que un año antes; los picos de cada año caen en mayo, diciembre y octubre
histograma  La mitad de los pedidos es de $360 o menos; 1 de cada 10 pasa de $795, casi siempre por un pastel entero
calor       38% de los pedidos llega después de las 7 de la noche; el pico es el jueves de 21 a 22 h
dispersion  Cada pedido extra en un día deja $416 más (r = 0.76): la venta depende de cuántos pedidos entran, no de pedidos más grandes
espectro    El ciclo fuerte es de 7.0 días (la semana); la quincena no aparece como pico (5% de la potencia semanal) ni se distingue del azar en venta
barras      Los pasteles enteros mandan todo el año; en diciembre venden 1.3 veces su promedio diario
dia_semana  El sábado vende 2.9 veces lo del martes ($604 contra $210); el lunes no hay entregas


## 5. Análisis espectral
Serie diaria por fecha de entrega con ceros en los días sin entregas, menos su media móvil de 28 días; `numpy.fft.rfft`, potencia |X(f)|² y periodo T = 1/f.

In [7]:
esp = an.espectro(an.serie_diaria(df))
an.picos(esp, n=6)

,frecuencia,periodo,potencia,potencia_relativa
0,0.14,7.01,"18,074,144.22",1.00
1,0.29,3.50,"7,330,201.99",0.41
2,0.05,19.62,"1,826,980.95",0.10
3,0.43,2.33,"1,542,672.45",0.09
4,0.48,2.06,"1,385,606.22",0.08
5,0.39,2.59,"1,364,049.38",0.08


In [8]:
pk = an.picos(esp)
print("potencia cerca de 15.2 días:", round(an.potencia_cerca(esp, 15.2) / pk.potencia.iloc[0], 3), "de la semanal")
print("potencia cerca de 30.4 días:", round(an.potencia_cerca(esp, 30.4) / pk.potencia.iloc[0], 3), "de la semanal")
an.prueba_quincena(df)

potencia cerca de 15.2 días: 0.047 de la semanal
potencia cerca de 30.4 días: 0.021 de la semanal


{'efecto': 0.07173697758860054,
 'p_valor': 0.40926342383198056,
 'dias_quincena': 259,
 'dias_normales': 1102}

La FFT encuentra la semana (7 días) y sus armónicos (3.5 y 2.33 días, que salen porque la semana no tiene forma de seno: el lunes vale cero y el sábado sube de golpe). La quincena no sobresale en el espectro porque su efecto es pequeño frente al ruido diario y porque no es exactamente periódica (los meses duran de 28 a 31 días). Comparando directamente quincena contra días normales del mismo día de la semana sale una diferencia chica que no es significativa: con estos datos no se puede confirmar el efecto de la quincena.

## 6. Recomendaciones

In [9]:
for x in an.recomendaciones(df):
    print(x["titulo"], "\n ", x["texto"], "\n")

Arma el fin de semana desde el jueves 
  El sábado vendes 2.9 veces lo de un martes, y para el jueves ya tienes apartados 69% de los pasteles enteros del sábado. Esa noche puedes dejar listos los cheesecakes que necesitan reposar y comprar el queso crema justo. Entre semana no hornees de más: en 58% de los martes y miércoles que abriste no hubo ninguna entrega. 

Los sábados fuertes ya llegan a tu tope: aparta con anticipo 
  En 6 de 51 sábados del último año entregaste 4 pasteles enteros, que es lo más que sale de tu cocina en un día. Esos días seguramente le dijiste que no a alguien. Abre la agenda del sábado el miércoles en tus historias y, cuando se llene, ofrece entrega el domingo. 

Contesta rápido de noche 
  35% de los pedidos llega después de las 7 de la noche. Publica el menú en historias por la tarde y deja respuestas rápidas en WhatsApp con precios para contestar al momento. La quincena no mueve tus ventas de forma clara (la diferencia que se ve es chica y puede ser casuali